In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("synthetic_fraud_dataset.csv")

In [3]:
df.shape

(10000, 10)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     10000 non-null  int64  
 1   user_id            10000 non-null  int64  
 2   amount             10000 non-null  float64
 3   transaction_type   10000 non-null  object 
 4   merchant_category  10000 non-null  object 
 5   country            10000 non-null  object 
 6   hour               10000 non-null  int64  
 7   device_risk_score  10000 non-null  float64
 8   ip_risk_score      10000 non-null  float64
 9   is_fraud           10000 non-null  int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 781.4+ KB


In [5]:
df.sample(5)

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
396,973,564,34.964912,QR,Food,TR,7,0.191183,0.115488,0
8044,2203,775,90.153714,ATM,Electronics,UK,16,0.222144,0.098273,0
3386,6357,505,168.738864,ATM,Food,US,14,0.007459,0.170670,0
4056,5631,941,176.880030,Online,Travel,UK,12,0.192520,0.062934,0
802,7280,366,147.728979,Online,Electronics,UK,11,0.037900,0.182454,0


In [6]:
X = df.drop(['transaction_id', 'user_id', 'is_fraud'], axis=1)
y = df['is_fraud']

In [7]:
df['merchant_category'].unique()

array(['Travel', 'Food', 'Clothing', 'Grocery', 'Electronics'],
      dtype=object)

In [8]:
df['transaction_type'].unique()

array(['ATM', 'QR', 'Online', 'POS'], dtype=object)

In [9]:
df['country'].unique()

array(['TR', 'US', 'FR', 'DE', 'UK', 'NG'], dtype=object)

In [10]:
X = pd.get_dummies(X, drop_first=True)

In [11]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   amount                         10000 non-null  float64
 1   hour                           10000 non-null  int64  
 2   device_risk_score              10000 non-null  float64
 3   ip_risk_score                  10000 non-null  float64
 4   transaction_type_Online        10000 non-null  bool   
 5   transaction_type_POS           10000 non-null  bool   
 6   transaction_type_QR            10000 non-null  bool   
 7   merchant_category_Electronics  10000 non-null  bool   
 8   merchant_category_Food         10000 non-null  bool   
 9   merchant_category_Grocery      10000 non-null  bool   
 10  merchant_category_Travel       10000 non-null  bool   
 11  country_FR                     10000 non-null  bool   
 12  country_NG                     10000 non-null  

In [12]:
X.head(1)

,amount,hour,device_risk_score,ip_risk_score,transaction_type_Online,transaction_type_POS,transaction_type_QR,merchant_category_Electronics,merchant_category_Food,merchant_category_Grocery,merchant_category_Travel,country_FR,country_NG,country_TR,country_UK,country_US
0,4922.587542,12,0.992347,0.947908,False,False,False,False,False,False,True,False,False,True,False,False


In [13]:
X = X.astype(int)

In [14]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

c:\Users\Parth Rana\Downloads\Financial-Fraud-Detection\ann\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [19]:
class_weight = {0:1, 1:5}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9320 - loss: 0.7359 - val_accuracy: 0.9830 - val_loss: 0.0624
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9617 - loss: 0.1895 - val_accuracy: 0.9505 - val_loss: 0.1241
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9740 - loss: 0.1368 - val_accuracy: 0.9680 - val_loss: 0.0767
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9720 - loss: 0.1467 - val_accuracy: 0.9660 - val_loss: 0.0960
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9803 - loss: 0.1095 - val_accuracy: 0.9830 - val_loss: 0.0532
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9726 - loss: 0.1453 - val_accuracy: 0.9840 - val_loss: 0.0492
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9800 - loss: 0.1053 - val_accuracy: 0.9790 - val_loss: 0.0489
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9815 - loss: 0.1075 - val_accu

In [20]:
from sklearn.metrics import classification_report

y_pred = (model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1887
           1       0.86      0.85      0.85       113

    accuracy                           0.98      2000
   macro avg       0.92      0.92      0.92      2000
weighted avg       0.98      0.98      0.98      2000



## Prediction

In [21]:
test_df = pd.DataFrame([
    [5000, 22, 0.8, 0.9, "Online", "Electronics", "US"],
    [100, 10, 0.1, 0.1, "ATM", "Food", "UK"],
    [2000, 18, 0.5, 0.4, "POS", "Grocery", "US"],
    [8000, 23, 0.9, 0.95, "Online", "Travel", "US"],
    [300, 14, 0.2, 0.2, "QR", "Clothing", "FR"]
], columns=[
    "amount", "hour", "device_risk_score", "ip_risk_score",
    "transaction_type", "merchant_category", "country"
])

In [22]:
test_df_encoded = pd.get_dummies(test_df)
test_df_encoded = test_df_encoded.reindex(columns=X.columns, fill_value=0)

# ✅ Apply scaling
test_scaled = scaler.transform(test_df_encoded)

# ✅ Predict using scaled data
probs = model.predict(test_scaled)

for i, prob in enumerate(probs):
    print(f"\nRow {i+1} → Probability: {prob[0]}")
    print("🚨 Fraud" if prob[0] > 0.5 else "✅ Safe")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step

Row 1 → Probability: 0.5270959138870239
🚨 Fraud

Row 2 → Probability: 0.1926761120557785
✅ Safe

Row 3 → Probability: 0.4628790020942688
✅ Safe

Row 4 → Probability: 0.5420464873313904
🚨 Fraud

Row 5 → Probability: 0.32369622588157654
✅ Safe
